# Lab 11: Chain-of-Thought

## Making the model show its work

The next cells build `build_direct_prompt` (asks for a short answer immediately) and `build_cot_prompt` (asks the model to reason step by step, then end with a `"Answer: <answer>"` line), plus `extract_final_answer` to pull that final line back out. See the Reading tab for what a reasoning trace is and why comparing a direct answer against a step-by-step one matters.

### Setup - get a free key at https://console.groq.com/keys

In [ ]:
"""Get your free Groq API key at https://console.groq.com/keys - it's never written to this notebook."""
import getpass
import json
import requests

GROQ_API_URL = "https://api.groq.com/openai/v1/chat/completions"
DEFAULT_MODEL = "llama-3.1-8b-instant"

GROQ_API_KEY = getpass.getpass("Enter your GROQ_API_KEY: ")


def build_request_payload(messages, model=DEFAULT_MODEL, temperature=0.7, response_format=None):
    """The request BODY: what we're asking the model to do."""
    payload = {"model": model, "messages": messages, "temperature": temperature}
    if response_format is not None:
        payload["response_format"] = response_format
    return payload


def extract_reply(response_json):
    """Pulls the assistant's text out of the RESPONSE body."""
    return response_json["choices"][0]["message"]["content"]


def call_groq(messages, api_key=None, model=DEFAULT_MODEL, temperature=0.7, response_format=None):
    """The full request: ENDPOINT + METHOD (POST) + HEADERS (auth) + BODY -> RESPONSE."""
    headers = {
        "Authorization": f"Bearer {api_key or GROQ_API_KEY}",
        "Content-Type": "application/json",
    }
    payload = build_request_payload(messages, model=model, temperature=temperature, response_format=response_format)
    response = requests.post(GROQ_API_URL, headers=headers, json=payload, timeout=30)
    response.raise_for_status()
    return extract_reply(response.json())


print("Groq setup ready. Call call_groq(messages) to send a chat completion request.")


### Concept 1 practice: comparing two approaches side by side
A quick blurt-it-out vs. show-your-work comparison on a different calculation.

In [ ]:
def guess_total(price, tax_rate):
    """Straight to an answer - fast, but nothing to check if it's wrong."""
    return price + price * tax_rate

def show_total_steps(price, tax_rate):
    """Break it into steps you can each verify."""
    tax = price * tax_rate
    total = price + tax
    return f"tax={tax:.2f}, total={total:.2f}"

print(guess_total(50, 0.08))        # expected output: 54.0
print(show_total_steps(50, 0.08))   # expected output: tax=4.00, total=54.00

### Concept 2 practice: pulling a value out of a larger string
The same "split on the last marker" trick, this time pulling a value out of a weather forecast.

In [ ]:
forecast = "Morning: cloudy\nAfternoon: sunny\nHigh: 78F"

def extract_high(text: str, marker: str = "High:") -> str:
    if marker not in text:
        return text.strip()
    return text.rsplit(marker, 1)[1].strip()

print(extract_high(forecast))  # expected output: 78F

## Exercise

Implement `build_direct_prompt`, `build_cot_prompt` (adds a "think step by step, then give your final answer as 'Answer: <answer>'" instruction), and `extract_final_answer` (pulls out everything after the marker).

In [ ]:
def build_direct_prompt(question: str) -> list[dict]:
    """TODO: a single user message asking the question directly, instructing
    a short final-answer-only reply."""
    raise NotImplementedError("build_direct_prompt() is not implemented yet - replace the TODO with your code")


def build_cot_prompt(question: str) -> list[dict]:
    """TODO: a single user message asking the model to think step by step,
    then end with a line 'Answer: <final answer>'."""
    raise NotImplementedError("build_cot_prompt() is not implemented yet - replace the TODO with your code")


def extract_final_answer(reply_text: str, marker: str = "Answer:") -> str:
    """TODO: return everything after the last occurrence of `marker`,
    stripped. If `marker` isn't present, return the whole reply, stripped."""
    raise NotImplementedError("extract_final_answer() is not implemented yet - replace the TODO with your code")


assert "step by step" in build_cot_prompt("What is 12 * 13?")[0]["content"].lower(), 'expected ' + repr("step by step") + ' to be in build_cot_prompt("What is 12 * 13?")[0]["content"].lower(), but it is ' + repr(build_cot_prompt("What is 12 * 13?")[0]["content"].lower())
assert extract_final_answer("Let's see... 12*13=156.\nAnswer: 156") == "156", 'extract_final_answer("Let\'s see... 12*13=156.\\nAnswer: 156") should be ' + repr("156") + ", got " + repr(extract_final_answer("Let's see... 12*13=156.\nAnswer: 156"))
assert extract_final_answer("no marker here") == "no marker here", 'extract_final_answer("no marker here") should be ' + repr("no marker here") + ", got " + repr(extract_final_answer("no marker here"))

print("All checks passed.")

## Solution

In [ ]:
def build_direct_prompt(question: str) -> list[dict]:
    return [{"role": "user", "content": f"{question} Answer in as few words as possible."}]


def build_cot_prompt(question: str) -> list[dict]:
    return [{
        "role": "user",
        "content": f"{question}\n\nThink step by step, then end your reply with a final "
                   "line in exactly this form: 'Answer: <your answer>'",
    }]


def extract_final_answer(reply_text: str, marker: str = "Answer:") -> str:
    if marker not in reply_text:
        return reply_text.strip()
    return reply_text.rsplit(marker, 1)[1].strip()


print(build_cot_prompt("What is 12 * 13?"))

## Real-world project (LIVE - needs your GROQ_API_KEY above)

Compares direct-answer vs. chain-of-thought prompting on a handful of small reasoning questions - the kind small/fast models often get wrong when rushed into a one-shot answer.

In [ ]:
QUESTIONS = [
    ("If a train travels 60 miles in 45 minutes, what is its speed in miles per hour?", "80"),
    ("A shirt costs $40 after a 20% discount. What was the original price?", "50"),
    ("Sam has 3 times as many apples as Jo. Together they have 24 apples. How many does Sam have?", "18"),
]

for question, expected in QUESTIONS:
    direct_reply = call_groq(build_direct_prompt(question))
    cot_reply = call_groq(build_cot_prompt(question))
    cot_answer = extract_final_answer(cot_reply)

    print(f"Q: {question}")
    print(f"  expected:            {expected}")
    print(f"  direct answer:       {direct_reply.strip()[:60]}")
    print(f"  chain-of-thought:    {cot_answer}")
    print()